In [3]:
import sklearn
import numpy as np
import pandas
import torch
import sklearn.datasets
import sklearn.preprocessing
import helpers
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, precision_score, recall_score, precision_recall_curve, roc_auc_score, f1_score, make_scorer, auc, average_precision_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Thyroid Dataset
Class 2 and 3 are Normal Data  
Class 1 is anomalous  
Analysis:
- Need to format these into 0 for normal data, 1 for anomalous
- 

In [46]:
X_thyroid, Y_thyroid = helpers.load_thyroid_dataset()

In [47]:
X_thyroid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 22 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       3772 non-null   float64
 1   1       3772 non-null   int64  
 2   2       3772 non-null   int64  
 3   3       3772 non-null   int64  
 4   4       3772 non-null   int64  
 5   5       3772 non-null   int64  
 6   6       3772 non-null   int64  
 7   7       3772 non-null   int64  
 8   8       3772 non-null   int64  
 9   9       3772 non-null   int64  
 10  10      3772 non-null   int64  
 11  11      3772 non-null   int64  
 12  12      3772 non-null   int64  
 13  13      3772 non-null   int64  
 14  14      3772 non-null   int64  
 15  15      3772 non-null   int64  
 16  16      3772 non-null   float64
 17  17      3772 non-null   float64
 18  18      3772 non-null   float64
 19  19      3772 non-null   float64
 20  20      3772 non-null   float64
 21  21      3772 non-null   int64  
dtype

In [48]:
X_thyroid.head()

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.73,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0.00060,0.015,0.120,0.082,0.146,3
1,0.24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.030,0.143,0.133,0.108,3
2,0.47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00190,0.024,0.102,0.131,0.078,3
3,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00090,0.017,0.077,0.090,0.085,3
4,0.23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.026,0.139,0.090,0.153,3


In [49]:
Y_thyroid

0       3
1       3
2       3
3       3
4       3
       ..
3767    3
3768    3
3769    2
3770    3
3771    3
Name: 21, Length: 3772, dtype: int64

In [50]:
Y_thyroid=Y_thyroid.where(Y_thyroid==1, 0)

In [51]:
Y_thyroid.value_counts()

21
0    3679
1      93
Name: count, dtype: int64

Insight: Severe class imbalance ~ 2.5% of anomalous data 

In [52]:
X_normal = X_thyroid[Y_thyroid==0].drop(X_thyroid.columns[-1], axis=1)
Y_normal = Y_thyroid[Y_thyroid==0]

X_hyperthyroids = X_thyroid[Y_thyroid == 1].drop(X_thyroid.columns[-1], axis=1)
Y_hyperthyroids = Y_thyroid[Y_thyroid == 1]

X_train_full, X_test, Y_train_full, Y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)
X_hyperthyroids_train, X_hyperthyroids_test, Y_hyperthyroids_train, Y_hyperthyroids_test = train_test_split(X_hyperthyroids, Y_hyperthyroids, test_size=0.2, random_state=42)
X_test = pandas.concat([X_test, X_hyperthyroids_test], ignore_index=True)
Y_test = pandas.concat([Y_test, Y_hyperthyroids_test], ignore_index=True)

X_train, X_validate, Y_train, Y_validate = train_test_split(X_train_full, Y_train_full, test_size=0.2, random_state=42)
X_validate = pandas.concat([X_validate, X_hyperthyroids_train], ignore_index=True)
Y_validate = pandas.concat([Y_validate, Y_hyperthyroids_train], ignore_index=True)


In [44]:
X_normal

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.73,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0.00060,0.0150,0.120,0.082,0.146
1,0.24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00025,0.0300,0.143,0.133,0.108
2,0.47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00190,0.0240,0.102,0.131,0.078
3,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00090,0.0170,0.077,0.090,0.085
4,0.23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00025,0.0260,0.139,0.090,0.153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3767,0.77,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00006,0.0206,0.125,0.107,0.117
3768,0.41,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00130,0.0250,0.125,0.114,0.109
3769,0.88,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.01300,0.0174,0.123,0.099,0.124
3770,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00078,0.0206,0.106,0.088,0.121


# OC_SVM from Sklearn: Anomaly Detection

In [53]:
oc_svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm_clf", OneClassSVM(kernel='rbf', gamma=0.01, nu=0.001))
])
oc_svm_clf.fit(X_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm_clf', OneClassSVM(gamma=0.01, nu=0.001))])

In [54]:
def output_formatter(predictions):
    predictions = np.where(predictions == 1, 0, 1)
    return predictions

In [55]:
predictions = output_formatter(oc_svm_clf.predict(X_validate))

In [56]:
Y_validate

0      0
1      0
2      0
3      0
4      0
      ..
658    1
659    1
660    1
661    1
662    1
Name: 21, Length: 663, dtype: int64

In [57]:
predictions

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [58]:
precision_score(Y_validate, predictions)

0.7307692307692307

In [59]:
recall_score(Y_validate, predictions)

0.5135135135135135

In [60]:
f1_score(Y_validate, predictions)

0.6031746031746031

Fine Tuning  

Scores = [(precision, recall, f1), ...]

In [61]:
from sklearn.model_selection import KFold
from itertools import product
gammas = ['scale', 'auto', 0.1, 0.001, 0.0001]
nus = [0.0001, 0.0005, 0.001, 0.01, 0.1]
models = []
avg_scores = []
kfold = KFold(n_splits=3, shuffle=True, random_state=42)
for (nu, gamma) in product(nus, gammas):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", OneClassSVM(kernel='rbf', gamma=gamma, nu=nu))
    ])
    models.append(model)
    p, r, f1=0,0,0
    for train_i, test_i in kfold.split(X_train_full):
        X_train_fold, X_test_fold = X_train_full.iloc[train_i], X_train_full.iloc[test_i]
        Y_train_fold, Y_test_fold = Y_train_full.iloc[train_i], Y_train_full.iloc[test_i]
        
        # Validate set to contain the anomalies
        X_test_fold = pandas.concat([X_test_fold, X_hyperthyroids_train], ignore_index=True)
        Y_test_fold = pandas.concat([Y_test_fold, Y_hyperthyroids_train], ignore_index=True)
        
        model.fit(X_train_fold)
        predictions = output_formatter(model.predict(X_test_fold))
        p += precision_score(Y_test_fold, predictions)
        r += recall_score(Y_test_fold, predictions)
        f1 += f1_score(Y_test_fold, predictions)
    
    avg_scores.append((p/3, r/3, f1/3))

avg_scores = np.array(avg_scores)

In [62]:
len(models)

25

In [63]:
avg_scores

array([[0.39913804, 0.88738739, 0.54966955],
       [0.41964524, 0.8963964 , 0.57088477],
       [0.17628065, 0.95945946, 0.29690631],
       [0.78869048, 0.2027027 , 0.32163743],
       [0.85026738, 0.2027027 , 0.32603476],
       [0.45348271, 0.86486486, 0.59435562],
       [0.45929773, 0.85135135, 0.59603169],
       [0.34452495, 0.91441441, 0.49976112],
       [0.80059524, 0.2027027 , 0.32275725],
       [0.82142857, 0.1981982 , 0.31770674],
       [0.45243645, 0.85585586, 0.59146823],
       [0.46299513, 0.85585586, 0.60018148],
       [0.34940902, 0.91441441, 0.50482982],
       [0.78869048, 0.2027027 , 0.32163743],
       [0.82142857, 0.1981982 , 0.31770674],
       [0.45831896, 0.86486486, 0.59869121],
       [0.46431083, 0.85585586, 0.60122802],
       [0.34539955, 0.91441441, 0.50077615],
       [0.75847962, 0.40540541, 0.52521854],
       [0.75138079, 0.38738739, 0.5096455 ],
       [0.36558633, 0.87387387, 0.51472021],
       [0.36925849, 0.87387387, 0.51851364],
       [0.

In [64]:
indices_promising_models = np.where(avg_scores[:,2]>0.6)[0]
avg_scores[avg_scores[:,2]>.6]

array([[0.46299513, 0.85585586, 0.60018148],
       [0.46431083, 0.85585586, 0.60122802]])

In [65]:
promising_models = [models[i] for i in indices_promising_models]

In [66]:
for model in promising_models:
    predictions = output_formatter(model.predict(X_validate))
    print(precision_score(Y_validate, predictions), recall_score(Y_validate, predictions), f1_score(Y_validate, predictions))

0.7126436781609196 0.8378378378378378 0.7701863354037267
0.7045454545454546 0.8378378378378378 0.7654320987654321


**Testing against test set**

In [67]:
for model in promising_models:
    predictions = output_formatter(model.predict(X_test))
    print(precision_score(Y_test, predictions), recall_score(
        Y_test, predictions), f1_score(Y_test, predictions))

0.2073170731707317 0.8947368421052632 0.33663366336633666
0.20987654320987653 0.8947368421052632 0.34


Performance significantly diminished, probably due to the unseen data in test set


Because of the existence of test set, we're gonna train on the full training set (only normal data)

In [68]:
X_normal = X_thyroid[Y_thyroid == 0].drop(X_thyroid.columns[-1], axis=1)
X_normal

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.73,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0.00060,0.0150,0.120,0.082,0.146
1,0.24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00025,0.0300,0.143,0.133,0.108
2,0.47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00190,0.0240,0.102,0.131,0.078
3,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00090,0.0170,0.077,0.090,0.085
4,0.23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00025,0.0260,0.139,0.090,0.153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3767,0.77,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00006,0.0206,0.125,0.107,0.117
3768,0.41,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00130,0.0250,0.125,0.114,0.109
3769,0.88,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.01300,0.0174,0.123,0.099,0.124
3770,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.00078,0.0206,0.106,0.088,0.121


In [69]:
X_thyroid_test, Y_thyroid_test = helpers.load_thyroid_testset()
Y_thyroid_test = Y_thyroid_test.where(Y_thyroid_test == 1, 0)
X_thyroid_test = X_thyroid_test.drop(X_thyroid_test.columns[-1], axis=1)

In [70]:
Y_thyroid_test.value_counts()

21
0    3355
1      73
Name: count, dtype: int64

In [71]:
X_thyroid_test.head()

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,0.29,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.0061,0.028,0.111,0.131,0.085
1,0.32,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.0013,0.019,0.084,0.078,0.107
2,0.35,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.0000,0.031,0.239,0.100,0.239
3,0.21,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.0010,0.018,0.087,0.088,0.099
4,0.22,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0.0004,0.022,0.134,0.135,0.099


In [72]:
from itertools import product
gammas = ['scale', 'auto', 0.1, 0.001, 0.0001]
nus = [0.0001, 0.0005, 0.001, 0.01, 0.1]
models = []
scores = []
for (nu, gamma) in product(nus, gammas):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm_clf", OneClassSVM(kernel='rbf', gamma=gamma, nu=nu))
    ])
    models.append(model)
    model.fit(X_normal)
    predictions = output_formatter(model.predict(X_thyroid_test))
    scores.append((precision_score(Y_thyroid_test, predictions), recall_score(
        Y_thyroid_test, predictions), f1_score(Y_thyroid_test, predictions)))
scores = np.array(scores)

In [73]:
scores

array([[0.175     , 0.8630137 , 0.29099307],
       [0.175     , 0.8630137 , 0.29099307],
       [0.10806452, 0.91780822, 0.19336219],
       [0.45945946, 0.23287671, 0.30909091],
       [0.45714286, 0.21917808, 0.2962963 ],
       [0.1875    , 0.82191781, 0.30534351],
       [0.1875    , 0.82191781, 0.30534351],
       [0.121673  , 0.87671233, 0.21368948],
       [0.45945946, 0.23287671, 0.30909091],
       [0.44444444, 0.21917808, 0.29357798],
       [0.18944099, 0.83561644, 0.30886076],
       [0.18944099, 0.83561644, 0.30886076],
       [0.12307692, 0.87671233, 0.2158516 ],
       [0.45945946, 0.23287671, 0.30909091],
       [0.45945946, 0.23287671, 0.30909091],
       [0.19003115, 0.83561644, 0.30964467],
       [0.19003115, 0.83561644, 0.30964467],
       [0.12307692, 0.87671233, 0.2158516 ],
       [0.36082474, 0.47945205, 0.41176471],
       [0.35106383, 0.45205479, 0.39520958],
       [0.11842105, 0.8630137 , 0.20826446],
       [0.11842105, 0.8630137 , 0.20826446],
       [0.

In [74]:
# Get indices of the best 3 models
indices = np.argsort(scores[:, 2])[-3:][::-1]
indices

array([18, 19, 16])

Pick the 3 best models

In [75]:
promising_models = [models[i] for i in indices]
promising_models_scores = [scores[i] for i in indices]

In [76]:
promising_models_scores

[array([0.36082474, 0.47945205, 0.41176471]),
 array([0.35106383, 0.45205479, 0.39520958]),
 array([0.19003115, 0.83561644, 0.30964467])]

In [84]:
promising_prediction = output_formatter(promising_models[0].predict(X_thyroid_test))

In [108]:
confusion_matrix(Y_thyroid_test, promising_prediction)

array([[3293,   62],
       [  38,   35]])

Recall in the context of Anomaly Detection, Confusion Matrix

| TN, FP |
| FN, TP |

In the context of correct diagnosis of hyperthyroids, perhaps false positives would be as important as true positives as we may not want healthy patients to be misdiagnosed

# Isolation Forest: Anomaly detection with Scikit Isolation Forest

In [119]:
from sklearn.ensemble import IsolationForest

X_train, X_test, Y_train, Y_test = train_test_split(X_normal, Y_normal, test_size=0.2, random_state=42)
X_test_with_anomalies = pandas.concat([X_test, X_hyperthyroids], ignore_index=True)
Y_test_with_anomalies = pandas.concat([Y_test, Y_hyperthyroids], ignore_index=True)

In [120]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2943 entries, 1609 to 3255
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       2943 non-null   float64
 1   1       2943 non-null   int64  
 2   2       2943 non-null   int64  
 3   3       2943 non-null   int64  
 4   4       2943 non-null   int64  
 5   5       2943 non-null   int64  
 6   6       2943 non-null   int64  
 7   7       2943 non-null   int64  
 8   8       2943 non-null   int64  
 9   9       2943 non-null   int64  
 10  10      2943 non-null   int64  
 11  11      2943 non-null   int64  
 12  12      2943 non-null   int64  
 13  13      2943 non-null   int64  
 14  14      2943 non-null   int64  
 15  15      2943 non-null   int64  
 16  16      2943 non-null   float64
 17  17      2943 non-null   float64
 18  18      2943 non-null   float64
 19  19      2943 non-null   float64
 20  20      2943 non-null   float64
dtypes: float64(6), int64(15)
memory usage: 

In [121]:
Y_hyperthyroids.info()

<class 'pandas.core.series.Series'>
Index: 93 entries, 19 to 3679
Series name: 21
Non-Null Count  Dtype
--------------  -----
93 non-null     int64
dtypes: int64(1)
memory usage: 1.5 KB


In [122]:
X_hyperthyroids.info()

<class 'pandas.core.frame.DataFrame'>
Index: 93 entries, 19 to 3679
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       93 non-null     float64
 1   1       93 non-null     int64  
 2   2       93 non-null     int64  
 3   3       93 non-null     int64  
 4   4       93 non-null     int64  
 5   5       93 non-null     int64  
 6   6       93 non-null     int64  
 7   7       93 non-null     int64  
 8   8       93 non-null     int64  
 9   9       93 non-null     int64  
 10  10      93 non-null     int64  
 11  11      93 non-null     int64  
 12  12      93 non-null     int64  
 13  13      93 non-null     int64  
 14  14      93 non-null     int64  
 15  15      93 non-null     int64  
 16  16      93 non-null     float64
 17  17      93 non-null     float64
 18  18      93 non-null     float64
 19  19      93 non-null     float64
 20  20      93 non-null     float64
dtypes: float64(6), int64(15)
memory usage: 16.0

In [123]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2943 entries, 1609 to 3255
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       2943 non-null   float64
 1   1       2943 non-null   int64  
 2   2       2943 non-null   int64  
 3   3       2943 non-null   int64  
 4   4       2943 non-null   int64  
 5   5       2943 non-null   int64  
 6   6       2943 non-null   int64  
 7   7       2943 non-null   int64  
 8   8       2943 non-null   int64  
 9   9       2943 non-null   int64  
 10  10      2943 non-null   int64  
 11  11      2943 non-null   int64  
 12  12      2943 non-null   int64  
 13  13      2943 non-null   int64  
 14  14      2943 non-null   int64  
 15  15      2943 non-null   int64  
 16  16      2943 non-null   float64
 17  17      2943 non-null   float64
 18  18      2943 non-null   float64
 19  19      2943 non-null   float64
 20  20      2943 non-null   float64
dtypes: float64(6), int64(15)
memory usage: 

In [124]:
Y_train.info()

<class 'pandas.core.series.Series'>
Index: 2943 entries, 1609 to 3255
Series name: 21
Non-Null Count  Dtype
--------------  -----
2943 non-null   int64
dtypes: int64(1)
memory usage: 46.0 KB


In [ ]:
contanmination_factors = [float(len(Y_hyperthyroids) / len(Y_normal)), 0.01, 0.001]
num_itrees = [10,15,20]
max_samples = [64, 'auto', 128]
max_features = [3,4,5,6]

In [126]:
from itertools import product
models = []
avg_scores = []
random_states = [40, 50, 60]
for (contanmination_factor, n_tree, max_sample, max_feature) in product(contanmination_factors, num_itrees, max_samples, max_features):
    model = IsolationForest(n_estimators=n_tree, contamination=contanmination_factor,
                            max_samples=max_sample, max_features=max_feature, n_jobs=-1, random_state=42)
    p, r, f1 = 0, 0, 0
    models.append(model)
    for rs in random_states:
        X_trainset, X_validate, Y_trainset, Y_validate = train_test_split(
            X_train, Y_train, test_size=0.2, random_state=rs)
        X_validate_with_anomalies = pandas.concat(
            [X_trainset, X_hyperthyroids], ignore_index=True)
        Y_validate_with_anomalies = pandas.concat(
            [Y_trainset, Y_hyperthyroids], ignore_index=True)

        model.fit(X_trainset)
        predictions = output_formatter(
            model.predict(X_validate_with_anomalies))

        p += precision_score(Y_validate_with_anomalies, predictions)
        r += recall_score(Y_validate_with_anomalies, predictions)
        f1 += f1_score(Y_validate_with_anomalies, predictions)

    avg_scores.append(
        [p/len(random_states), r/len(random_states), f1/len(random_states)])

In [127]:
avg_scores = np.array(avg_scores)
avg_scores

array([[0.05722705, 0.03942652, 0.04668433],
       [0.28523416, 0.25806452, 0.27092699],
       [0.2668987 , 0.25806452, 0.26114098],
       [0.36022386, 0.40501792, 0.37935513],
       [0.08100192, 0.05734767, 0.06714441],
       [0.41420787, 0.46236559, 0.43631384],
       [0.28985993, 0.2688172 , 0.27850877],
       [0.3947903 , 0.46236559, 0.42336449],
       [0.07592204, 0.05376344, 0.06293194],
       [0.32278364, 0.31541219, 0.31833595],
       [0.27420242, 0.26164875, 0.26644961],
       [0.39159587, 0.4265233 , 0.40683693],
       [0.26345275, 0.23297491, 0.24714481],
       [0.53001969, 0.72759857, 0.61328671],
       [0.3281125 , 0.31541219, 0.32160846],
       [0.56624686, 0.84229391, 0.67675988],
       [0.48823349, 0.62365591, 0.54691249],
       [0.56093891, 0.82437276, 0.66759488],
       [0.49641663, 0.63799283, 0.55814767],
       [0.54986713, 0.7921147 , 0.64870742],
       [0.36580775, 0.37992832, 0.37198198],
       [0.51206816, 0.69175627, 0.58692985],
       [0.

In [128]:
indices = np.argsort(avg_scores[:, 2])[-3:][::-1]
indices

array([51, 23, 15])

In [129]:
best_models = [models[i] for i in indices]
best_models

[IsolationForest(contamination=0.01, max_features=6, max_samples=64,
                 n_estimators=15, n_jobs=-1, random_state=42),
 IsolationForest(contamination=0.025278608317477577, max_features=6,
                 max_samples=128, n_estimators=15, n_jobs=-1, random_state=42),
 IsolationForest(contamination=0.025278608317477577, max_features=6,
                 max_samples=64, n_estimators=15, n_jobs=-1, random_state=42)]

In [130]:
best_if_model = best_models[0]

In [131]:
pred = output_formatter(best_if_model.predict(X_test_with_anomalies))
[precision_score(Y_test_with_anomalies, pred), recall_score(
    Y_test_with_anomalies, pred), f1_score(Y_test_with_anomalies, pred)]

[0.8987341772151899, 0.7634408602150538, 0.8255813953488372]

In [132]:
confusion_matrix(Y_test_with_anomalies, pred)

array([[728,   8],
       [ 22,  71]])

In [133]:
scores = []
for rs in range(1, 100):
    X_trainset, X_test, Y_trainset, Y_test = train_test_split(
        X_normal, Y_normal, test_size=0.2, random_state=rs)
    X_test_with_anomalies = pandas.concat(
        [X_test, X_hyperthyroids], ignore_index=True)
    Y_test_with_anomalies = pandas.concat(
        [Y_test, Y_hyperthyroids], ignore_index=True)
    best_if_model.fit(X_trainset)
    predictions = output_formatter(
        best_if_model.predict(X_test_with_anomalies))

    scores.append([precision_score(Y_test_with_anomalies, predictions), recall_score(
        Y_test_with_anomalies, predictions), f1_score(Y_test_with_anomalies, predictions)])

In [134]:
scores = np.array(scores)
np.mean(scores, axis=0)

array([0.87902203, 0.62126643, 0.71952331])

In [135]:
scores_feature_scaled = []
iso_forest = Pipeline([
    ("scaler", StandardScaler()),
    ("if_clf", best_if_model)
])
for rs in range(1, 100):
    X_trainset, X_validate, Y_trainset, Y_test = train_test_split(
        X_normal, Y_normal, test_size=0.2, random_state=rs)
    X_test_with_anomalies = pandas.concat(
        [X_test, X_hyperthyroids], ignore_index=True)
    Y_test_with_anomalies = pandas.concat(
        [Y_test, Y_hyperthyroids], ignore_index=True)
    iso_forest.fit(X_trainset)
    predictions = output_formatter(
        iso_forest.predict(X_test_with_anomalies))

    scores_feature_scaled.append([precision_score(Y_test_with_anomalies, predictions), recall_score(
        Y_test_with_anomalies, predictions), f1_score(Y_test_with_anomalies, predictions)])

In [136]:
scores_feature_scaled = np.array(scores_feature_scaled)
np.mean(scores_feature_scaled, axis=0)

array([0.89074217, 0.62126643, 0.72443971])

Conclusion: Scaled performance slightly worst

In [107]:
best_if_model.get_params()

{'bootstrap': False,
 'contamination': 0.01,
 'max_features': 6,
 'max_samples': 64,
 'n_estimators': 15,
 'n_jobs': -1,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

Current model performances:  
    - OC-SVM( ? ): F1_score ~ 0.41176471   
    - Isolation Forest ('contamination': 0.01, 'max_features': 6, 'max_samples': 64, 'n_estimators': 15, 'n_jobs': -1, ) : F1_score ~ 0.72443971

# Autoencoder: In the context of Anomaly Detection